# Assignment #6: Clickbait analysis

## Introduction

Your task for the final assignment will be to run a classification task using Hugging Face model. This is something that is very useful in practice, you get the model well suited for the task, then you apply it to your data. This could be anything, starting from, say, employees satisfaction scores to classifying customer emails for further processing.

## Dataset general information

You will use for this exercise a synthetic dataset, you can try to apply it to real datasets (which are very big), for instance:

- UCI News Aggregator Dataset - https://archive.ics.uci.edu/dataset/359/news+aggregator
- All the News Dataset - https://www.kaggle.com/datasets/davidmckinley/all-the-news-dataset

In our dataset:
- headlines were generated artificially
- some patterns are intentionally exaggerated
- some source or category biases may be stronger than in real data

Your task is not only to run the model, but also to question the patterns you observe.


## Submission
- Create your copy of this Notebook, put in the Google Colab, create share link and put this link on Kampus on Assignment #5 homework. Please check if anyone with the link can access your notebook! 

## Requirements

* Provide your homework timely, by putting working link on Kampus (late deliveries get less points)!
* Make sure your code is commented, so it is clear what you want to achieve. Some of the tasks contain question to answer, **make sure you will provide your answers!**


## Data Analysis Project with Hugging Face and Pandas

### Dataset
Use the file `headlines_dataset_2500_biased.csv`.

The dataset contains synthetic news headlines with intentional patterns, noise, and bias.
It is designed for exploratory analysis, feature engineering, and model-based classification.

### Columns
- `headline` — headline text
- `source` — news source / publisher
- `category` — one of: `politics`, `sports`, `entertainment`, `health`, `tech`
- `date` — publication date
- `id` — URL-like identifier or simple id





## Task 10. Reflection

Write a short conclusion answering:
1. Which sources appear most clickbait-heavy?
2. Which categories are most associated with clickbait?
3. What linguistic features seem most useful?
4. Did you notice time-based patterns?
5. Did the model make mistakes or surprising predictions?
6. Was the rule-based baseline useful?





In [1]:
################## Solutions #####################

## Task 1. Load and inspect the dataset

1. Load the CSV file into a Pandas DataFrame.
2. Display the first 5 rows.
3. Check:
   - shape of the dataset
   - data types
   - missing values
   - number of unique sources
4. Convert the `date` column to datetime (replace column)
5. Calculate the following statistics
    - How many headlines are there in total?
    - How many headlines are there in each category?
    - Which sources appear most often?

In [2]:
import pandas as pd 
#1
df = pd.read_csv("headlines_dataset_2500_biased.csv")
#2
df = pd.DataFrame(df)
df.head(5)
#3
print(f"Shape of the dataset: {df.shape}\n")
print(f"data types: {df.info()}\n")
print(f"missing values: {df.isnull().sum()}\n")
print(f"number of unique sources: {df["source"].drop_duplicates().count()}\n")
#4
df["date"] = pd.to_datetime(df["date"])
df.info()
#5
print(f"How many headlines are there in total? - {df["headline"].drop_duplicates().count()}\n")
print(f"How many headlines are there in each category? - \n{df.groupby("category")["headline"].nunique()}\n")
print(f"Which sources appear most often? - \n{df["source"].value_counts()}\n")

Shape of the dataset: (2500, 5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   headline  2500 non-null   object
 1   source    2500 non-null   object
 2   category  2500 non-null   object
 3   date      2500 non-null   object
 4   id        2500 non-null   object
dtypes: object(5)
memory usage: 97.8+ KB
data types: None

missing values: headline    0
source      0
category    0
date        0
id          0
dtype: int64

number of unique sources: 25

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   headline  2500 non-null   object        
 1   source    2500 non-null   object        
 2   category  2500 non-null   object        
 3   date      2500 non-null   datetime64[ns]
 4   id        2500 non-

## Task 2. Create text features with Pandas

Create the following columns:

- `headline_length_chars`
- `headline_length_words`
- `num_exclamation_marks`
- `num_question_marks`
- `contains_you` (you as a word, which might suggest clickbait)
- `has_number`
- `num_uppercase_words`
- `avg_word_length`

Hint, as we haven't covered string operations in Pandas fully :)

```python
import numpy as np

df["headline_length_chars"] = df["headline"].str.len()
df["headline_length_words"] = df["headline"].str.split().str.len()
df["num_exclamation_marks"] = df["headline"].str.count("!")
df["num_question_marks"] = df["headline"].str.count(r"\?")
df["contains_you"] = df["headline"].str.contains(r"\byou\b", case=False, regex=True)
df["has_number"] = df["headline"].str.contains(r"\d", regex=True)

df["num_uppercase_words"] = df["headline"].apply(
    lambda x: sum(word.isupper() and len(word) > 1 for word in str(x).split())
)

df["avg_word_length"] = df["headline"].apply(
    lambda x: np.mean([len(w) for w in str(x).split()]) if str(x).split() else 0
)
```

In [3]:
import numpy as np

df["headline_length_chars"] = df["headline"].str.len()
df["headline_length_words"] = df["headline"].str.split().str.len()
df["num_exclamation_marks"] = df["headline"].str.count("!")
df["num_question_marks"] = df["headline"].str.count(r"\?")
df["contains_you"] = df["headline"].str.contains(r"\byou\b", case=False, regex=True)
df["has_number"] = df["headline"].str.contains(r"\d", regex=True)

df["num_uppercase_words"] = df["headline"].apply(
    lambda x: sum(word.isupper() and len(word) > 1 for word in str(x).split())
)

df["avg_word_length"] = df["headline"].apply(
    lambda x: np.mean([len(w) for w in str(x).split()]) if str(x).split() else 0
)
df

,headline,source,category,date,id,headline_length_chars,headline_length_words,num_exclamation_marks,num_question_marks,contains_you,has_number,num_uppercase_words,avg_word_length
0,cloud provider releases update on pricing update,TechWorld,tech,2024-05-13,https://example-news.local/tech/2024/05/13/tec...,48,7,0,0,False,False,0,6.000000
1,rookie guard publishes report on training report,Arena Report,sports,2025-03-06,id_sports-0809,48,7,0,0,False,False,0,6.000000
2,president outlines plan for campaign strategy,National Wire,politics,2025-08-26,id_politics-0037,45,6,0,0,False,False,0,6.666667
3,captain responds to questions about comeback bid,GoalLine Daily,sports,2025-01-08,https://example-news.local/sports/2025/01/08/s...,48,7,0,0,False,False,0,6.000000
4,You won't believe what club owners said about ...,GoalLine Daily,sports,2025-12-17,id_sports-0943,58,10,0,0,True,False,0,4.900000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2495,AI lab discusses server outage during an inves...,Digital Chronicle,tech,2024-08-12,id_tech-2154,54,9,0,0,False,False,1,5.111111
2496,You won't believe what national team said abou...,Victory Times,sports,2025-02-06,id_sports-0547,60,10,0,0,True,False,0,5.100000
2497,city council reviews progress on transport bill,National Wire,politics,2024-10-21,https://example-news.local/politics/2024/10/21...,47,7,0,0,False,False,0,5.857143
2498,robotics company outlines plan for security flaw,TechWorld,tech,2025-12-16,https://example-news.local/tech/2025/12/16/tec...,48,7,0,0,False,False,0,6.000000


## Task 3. Classify headlines with a Hugging Face model

Use a zero-shot classifier. You can use whatever model you like, make sure you know how to call your model. Also remember that first run will be slow, as you need to download a model.

```python
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

labels = ["clickbait", "neutral news", "opinion", "sensational", "misleading"]
```

This classifier expects as arguments: 1. your headling and 2. list of suggested labels (given above, but you can experiment). So, as an example, if you call your classifier for the first headline

```python
result = classifier(df['headline'].iloc[0], labels)

```

you will get back the following response: 

```python
{'sequence': 'cloud provider releases update on pricing update', 'labels': ['sensational', 'misleading', 'neutral news', 'clickbait', 'opinion'], 'scores': [0.30910342931747437, 0.26624685525894165, 0.165482297539711, 0.14100110530853271, 0.11816634982824326]}
```

it returns a headline, labels and assigned to them scores. As you see this is sorted by the score, from highest to lowest.

Now, your task is to write a function that returns:
- top predicted label
- top score
- score for `"clickbait"`

Hint: it is a good idea to return from the function a Pandas series...

```python
def classify_headline(text):
    result = classifier(text, labels)
    return pd.Series({
        "predicted_label": result["labels"][0],
        "top_score": result["scores"][0],
        "clickbait_score": result["scores"][result["labels"].index("clickbait")]
    })
```

...because you can nicely add those data as 3 additional columns to your Pandas DataFrame:

```python
df[["predicted_label", "top_score", "clickbait_score"]] = df["headline"].apply(classify_headline)
```

Recall, that `apply` wants a function of a single argument that will be applied to every element of the selected column (headline in our case)

> Tip: first test your code on a sample of 200 rows to check if this works reasonably, then run it on the full dataset.

At this point we have our DataFrame with all the needed information.


In [4]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

labels = ["clickbait", "neutral news", "opinion", "sensational", "misleading"]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [5]:
result = classifier(df['headline'].iloc[0], labels)
result


{'sequence': 'cloud provider releases update on pricing update',
 'labels': ['sensational',
  'misleading',
  'neutral news',
  'clickbait',
  'opinion'],
 'scores': [0.3091033101081848,
  0.2662469446659088,
  0.1654830127954483,
  0.14100100100040436,
  0.1181657686829567]}

In [13]:
def classify_headline(text):
    result = classifier(text, labels)
    return pd.Series({
        "predicted_label": result["labels"][0],
        "top_score": result["scores"][0],
        "clickbait_score": result["scores"][result["labels"].index("clickbait")]
    })

sample = df.head(200).copy()

sample[["predicted_label", "top_score", "clickbait_score"]] = (
    sample["headline"].apply(classify_headline)
)



## Task 4. Analyze the dataset by source

Create a summary table by `source` (so, `groupby` by source) with:
- number of headlines
- average clickbait score
- average number of words
- average number of exclamation marks

Then answer the following questions
- Which sources appear most clickbait-heavy?
- Are there sources with many headlines but low clickbait scores?
- Which source uses the most punctuation?

In [31]:
source = sample.groupby("source").agg(
    number_of_headlines=("headline", "count"),
    avg_clickbait_score=("clickbait_score", "mean"),
    avg_number_of_words=("headline_length_words", "mean"),
    avg_exclamation_marks=("num_exclamation_marks", "mean")
).sort_values("avg_clickbait_score", ascending=False).reset_index()

source


,source,number_of_headlines,avg_clickbait_score,avg_number_of_words,avg_exclamation_marks
0,Victory Times,6,0.182571,8.500000,0.166667
1,HealthLine,11,0.165852,8.181818,0.000000
2,Arena Report,9,0.164216,9.111111,0.111111
3,National Wire,10,0.158144,8.400000,0.000000
4,Code & Circuit,8,0.155393,7.000000,0.000000
5,Wellness Journal,12,0.145955,8.500000,0.083333
6,Public Affairs Today,10,0.145618,9.000000,0.200000
7,Daily Ledger,11,0.139120,9.909091,0.181818
8,MatchDay Live,6,0.126749,9.333333,0.333333
9,Medical Brief,6,0.123445,8.000000,0.000000


**Task 4 question answers**

1. Victory Times gets the highest clickbait score
2. Yes,For examle Buzznow and Starflash 
3. Future Stack uses the most punctuation



## Task 5. Analyze the dataset by category

Create a summary table by `category` with:
- number of headlines
- average clickbait score
- average headline length
- proportion containing the word `"you"`

Answer the following questions
- Which category seems most clickbait-oriented?
- Which category has the longest headlines?



In [32]:
source = sample.groupby("category").agg(
    number_of_headlines=("headline", "count"),
    avg_clickbait_score=("clickbait_score", "mean"),
    avg_headline_length=("headline_length_chars", "mean"),
    proportion_contains_you=("contains_you", "mean")
).sort_values("avg_clickbait_score", ascending=False).reset_index()

source


,category,number_of_headlines,avg_clickbait_score,avg_headline_length,proportion_contains_you
0,health,34,0.135929,58.705882,0.000000
1,politics,45,0.134380,57.822222,0.022222
2,sports,41,0.127125,55.829268,0.048780
3,tech,35,0.125340,57.514286,0.000000
4,entertainment,45,0.082480,62.466667,0.044444


**Task 5 questions answers**
1. health
2. entertaiment



## Task 6. Time analysis

Create:
- `year`
- `month`
- `day_name`
- `week`

Hint:

```python
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day_name"] = df["date"].dt.day_name()
df["week"] = df["date"].dt.isocalendar().week
```

Then analyze:
1. Average clickbait score by month
2. Average clickbait score by day of week

Answer the following questions:
- Are weekends more clickbait-heavy than weekdays?
- Which month has the highest average clickbait score?



In [34]:
sample["year"] = sample["date"].dt.year
sample["month"] = sample["date"].dt.month
sample["day_name"] = sample["date"].dt.day_name()
sample["week"] = sample["date"].dt.isocalendar().week

x = sample.groupby("month").agg(
    avg_clickbait_score=("clickbait_score", "mean"),
    number_of_headlines=("headline", "count")
).sort_values("avg_clickbait_score", ascending=False).reset_index()
y = sample.groupby("day_name").agg(
    avg_clickbait_score=("clickbait_score", "mean"),
    number_of_headlines=("headline", "count")
).sort_values("avg_clickbait_score", ascending=False).reset_index()
y
x




,month,avg_clickbait_score,number_of_headlines
0,4,0.150616,14
1,9,0.147854,18
2,12,0.147583,17
3,2,0.139032,20
4,10,0.132317,13
5,8,0.121416,24
6,3,0.114922,15
7,7,0.113966,15
8,11,0.105360,14
9,5,0.098233,15


 **Task 6 questions answers**
1. No, weekends are not more clickbait-heavy than weekdays, because Thursday and Friday have higher average clickbait scores than the weekend days.
2. April

## Task 7. Compare clickbait vs non-clickbait headlines

Create:

```python
df["is_clickbait"] = df["predicted_label"] == "clickbait"
```

Then compare clickbait vs non-clickbait headlines using groupby:
- average character length
- average word count
- average punctuation
- proportion with numbers
- proportion containing `"you"`

Answer the following questions:
- Are clickbait headlines longer or shorter?
- Do clickbait headlines contain more punctuation?
- Do clickbait headlines more often contain `"you"` or numbers?


In [19]:
sample["is_clickbait"] = sample["predicted_label"] == "clickbait"
clickbait_comparison = sample.groupby("is_clickbait").agg(
    avg_character_length=("headline_length_chars", "mean"),
    avg_word_count=("headline_length_words", "mean"),
    avg_exclamation_marks=("num_exclamation_marks", "mean"),
    avg_question_marks=("num_question_marks", "mean"),
    proportion_has_number=("has_number", "mean"),
    proportion_contains_you=("contains_you", "mean")
)

clickbait_comparison

,avg_character_length,avg_word_count,avg_exclamation_marks,avg_question_marks,proportion_has_number,proportion_contains_you
is_clickbait,,,,,,
False,58.520408,8.882653,0.122449,0.030612,0.076531,0.02551
True,60.250000,9.250000,0.500000,0.000000,0.250000,0.00000


**Task 7 questions answers**
1. Clickbait headlines are slightly longer.
2. Clickbait headlines contain more exclamation marks, but fewer question marks.
3. Clickbait headlines more often contain numbers

## Task 8. Ranking and filtering

Find:
1. The 10 headlines with the highest `clickbait_score`
2. The 10 headlines with the lowest `clickbait_score`
3. The most ambiguous headlines, meaning the lowest `top_score`

Hint:

```python
most_clickbait = df.sort_values("clickbait_score", ascending=False).head(10)
least_clickbait = df.sort_values("clickbait_score", ascending=True).head(10)
ambiguous = df.sort_values("top_score").head(10)
```

And, as usually, some questions:
- Do the top-ranked clickbait headlines really look like clickbait?
- Which ambiguous headlines are hard to classify?
- Are there suspicious or surprising cases?

In [48]:
most_clickbait = sample.sort_values("clickbait_score", ascending=False).head(10)

least_clickbait = sample.sort_values("clickbait_score", ascending=True).head(10)
#least_clickbait
ambiguous = sample.sort_values("top_score").head(10)
#ambiguous
most_clickbait["headline"].to_list()
ambiguous["headline"].to_list()


['sleep clinic releases update on hydration guidance',
 'city council releases update on public sector wages',
 'city council outlines plan for campaign strategy',
 'hospital network releases update on workout habit',
 'budget office reviews progress on budget proposal this weekend',
 'public health agency announces changes to diet trend',
 'budget office publishes report on campaign strategy',
 'rookie guard publishes report on training report',
 'chipmaker confirms new details about feature rollout',
 'parliament outlines plan for ethics inquiry']

**Task 8 questions answers**
1. Yes, because "The internet is obsessed with midfielder's latest comeback bid!" sounds like clickbait
2. They are hard to classify because they look neutral and do not have clear clickbait signals
3. I guess no, all of them litarlly look like clickbait 

## Task 9. Build a simple rule-based baseline

```python
df["rule_based_score"] = (
    0.2 * df["contains_you"].astype(int) +
    0.2 * df["has_number"].astype(int) +
    0.2 * (df["num_exclamation_marks"] > 0).astype(int) +
    0.2 * (df["num_question_marks"] > 0).astype(int) +
    0.2 * (df["num_clickbait_words"] > 0).astype(int)
)
```

Then compare it with the model:
- correlation between `rule_based_score` and `clickbait_score`
- headlines where the baseline and the model disagree strongly

Questions:
- Does the simple baseline roughly agree with the model?
- Where does the baseline fail?
- Where does the model perhaps overreact?


In [50]:
sample["num_clickbait_words"] = len(sample["headline"])
sample["rule_based_score"] = (
    0.2 * sample["contains_you"].astype(int) +
    0.2 * sample["has_number"].astype(int) +
    0.2 * (sample["num_exclamation_marks"] > 0).astype(int) +
    0.2 * (sample["num_question_marks"] > 0).astype(int) +
    0.2 * (sample["num_clickbait_words"] > 0).astype(int)
)
correlation = sample["rule_based_score"].corr(sample["clickbait_score"])
correlation


np.float64(-0.00915878043890062)

In [52]:
sample["difference"] = (
    sample["clickbait_score"] - sample["rule_based_score"]
).abs()
sample.sort_values("difference", ascending=False).head(10)

,headline,source,category,date,id,headline_length_chars,headline_length_words,num_exclamation_marks,num_question_marks,contains_you,...,top_score,clickbait_score,year,month,day_name,week,is_clickbait,num_clickbait_words,rule_based_score,difference
169,5 reasons the shocking truth about AI assistan...,Future Stack,tech,2025-03-16,https://example-news.local/tech/2025/03/16/tec...,73,12,1,0,False,...,0.906595,0.013287,2025,3,Sunday,11,False,200,0.6,0.586713
104,12 reasons what happened to startup over secur...,Future Stack,tech,2024-09-12,https://example-news.local/tech/2024/09/12/tec...,76,12,1,0,False,...,0.874501,0.015037,2024,9,Thursday,37,False,200,0.6,0.584963
135,7 reasons the surprising reason actor changed ...,Daily Ledger,entertainment,2024-12-23,https://example-news.local/entertainment/2024/...,95,17,1,0,False,...,0.838282,0.026277,2024,12,Monday,52,False,200,0.6,0.573723
148,10 reasons the surprising reason phone maker c...,TechWorld,tech,2025-10-05,id_tech-2484,79,12,2,0,False,...,0.875083,0.028938,2025,10,Sunday,40,False,200,0.6,0.571062
17,WATCH: Is this the wildest surprise cameo stor...,PopScope,entertainment,2025-06-10,https://example-news.local/entertainment/2025/...,61,11,1,1,False,...,0.847370,0.067484,2025,6,Tuesday,24,False,200,0.6,0.532516
178,You won't believe what coach said about traini...,Arena Report,sports,2025-08-01,https://example-news.local/sports/2025/08/01/s...,56,9,1,0,True,...,0.757114,0.100687,2025,8,Friday,31,False,200,0.6,0.499313
66,You won't believe what foreign affairs team sa...,Public Affairs Today,politics,2024-07-23,https://example-news.local/politics/2024/07/23...,75,12,1,0,True,...,0.648239,0.174372,2024,7,Tuesday,30,False,200,0.6,0.425628
180,Is this the wildest playoff push story of the ...,MatchDay Live,sports,2024-10-20,https://example-news.local/sports/2024/10/20/s...,52,10,1,1,False,...,0.619051,0.178538,2024,10,Sunday,42,False,200,0.6,0.421462
141,5 reasons experts are stunned by this new awar...,BuzzNow,entertainment,2024-06-20,id_entertainment-1378,84,16,0,0,False,...,0.741655,0.013195,2024,6,Thursday,25,False,200,0.4,0.386805
110,What happened to studio executive over final e...,ScreenBeat,entertainment,2024-02-16,https://example-news.local/entertainment/2024/...,98,16,2,0,False,...,0.800254,0.013329,2024,2,Friday,7,False,200,0.4,0.386671


**Task 9 questions answers**

1. No, the baseline does not fully agree with the model.
2. The baseline fails because it gives high scores for simple signals like numbers and exclamation marks.
3. The model may underreact to some headlines that look clearly clickbait-like.

### Task 10 (no code :))

Write a short conclusion answering:
1. Which sources appear most clickbait-heavy?
2. Which categories are most associated with clickbait?
3. What linguistic features seem most useful?
4. Did you notice time-based patterns?
5. Did the model make mistakes or surprising predictions?
6. Was the rule-based baseline useful?





1. Victory Times gets the highest clickbait score
2. Healt because it gets the highest clickbait score
3. The most useful linguistic features seem to be the numbers and exclamation
4. At the end of the weekdays, especially on Thursday and Friday, clickbait appears more often. This may be because people use social media more at that time. However, there isn't clear trend by month
5. I didn't noticed any mistakes or suprising predictions
6. The rule-based baseline was useful as a simple comparison, but it often failed. In these examples, it gave a high score because of numbers, exclamation marks and clickbait words, while the model did not classify them as clickbait. This shows that the baseline does not understand context well. Maybe i should adjust weight appropriately

## Task 11. Optional (!) advanced extensions

### A. Visualizations
Create:
- bar chart of clickbait rate by source
- histogram of clickbait scores

### B. Model comparison
Compare two Hugging Face models and see whether their predictions differ.

### C. Label wording experiment
Try different candidate label sets.

```python
labels_1 = ["clickbait", "neutral news", "sensational"]
labels_2 = ["misleading headline", "standard report", "attention-grabbing headline"]
```

In [ ]:
# Task 11 (Optional)

